# K2460 — one notebook per run

`BACKEND = "simulator"` runs a synthetic cell (not bench validated);
`BACKEND = "k2460"` runs the Keithley 2460 through the packaged sequencer
runtime. The settings cell is this run's environment: set `BACKEND`, the
instrument and `LIMITS` there only for the cell and wiring actually connected.
Each run is validated against `LIMITS`; there is no further confirmation.
Review the run-sheet row and preview before running. Install the project with the `experiment` and `visa` extras and
use a Jupyter kernel from that environment.

Keep `DATA_DIR` on **local disk**: HDF5 uses SWMR for live readers. Create this
notebook in that same directory, then save it after execution so the notebook
and HDF5 file form the human and machine records of this run. Use `SIM_SPEED = 1.0`
for realistic pacing; 10.0 is an accelerated demo. In this template the limits are
dummy-cell placeholders. A copy made with `oe-new-run` instead
holds the session limits the whole sheet was validated against, and writes its HDF5
file next to itself (`DATA_DIR`).
Create a workbook with `make_template` from `sequences.run_sheet` if needed.

In [ ]:
from pathlib import Path

from ophyd_electrochemistry.sequences import SafetyLimits

MEASUREMENT_ID = "example-rest"
RUN_SHEET = Path("runs.xlsx")
DATA_DIR = Path.home() / "k2460-data"
# DUMMY-CELL VALUES ONLY — review voltage, current, duration and record bounds.
LIMITS = SafetyLimits(
    v_min_abs=0.0,
    v_max_abs=3.0,
    i_max_abs=0.01,
    max_duration_s=200000.0,
    max_records=1000000,
)
BACKEND = "simulator"  # or "k2460"
SIM_SPEED = 10.0
# Hardware only; bench values of 3 Nov 2026, reconfirm before use.
VISA_RESOURCE = "TCPIP0::169.254.113.151::5025::SOCKET"
EXPECTED_SERIAL = "04686198"
TERMINALS = "front"
START_TIMEOUT_S = None  # external START: abort if none arrives within this many seconds

## Select the run
The entire sheet is validated against these session limits before selection.

In [ ]:
from ophyd_electrochemistry.exceptions import ValidationError
from ophyd_electrochemistry.sequences import SequenceRequest
from ophyd_electrochemistry.sequences.run_sheet import load_run_sheet

runs = load_run_sheet(RUN_SHEET, LIMITS)
for run_id, spec in runs.items():
    selected = " <-- selected" if run_id == MEASUREMENT_ID else ""
    print(f"{run_id}: {spec.row['protocol']}{selected}")
if MEASUREMENT_ID not in runs:
    raise ValidationError(
        f"Unknown measurement_id {MEASUREMENT_ID!r}; available IDs: {', '.join(runs)}"
    )
request = SequenceRequest.from_run_spec(runs[MEASUREMENT_ID], LIMITS)

## Review
Expected records and duration are full-duration upper bounds; cutoffs can end steps early.

In [ ]:
from ophyd_electrochemistry.sequences.notebook import preview_figure, run_summary

print(run_summary(request))
preview = preview_figure(request)
preview.show()

## Device
`BACKEND` selects the backend. Nothing is sent to the 2460 here: the run cell
connects, installs the packaged runtime with the output OFF, uploads and arms.
For an external-start row, the simulator supplies a synthetic START after 0.5 s;
the 2460 waits for the START line (input 6) with READY (output 1) high.

In [ ]:
from ophyd_electrochemistry.exceptions import ValidationError
from ophyd_electrochemistry.keithley.k2460.sequence_backend import k2460_sequence_backend
from ophyd_electrochemistry.sequences import SequenceFlyer, SimulatedSequenceBackend

if BACKEND == "simulator":
    backend = SimulatedSequenceBackend(
        speed=SIM_SPEED,
        external_start_delay_s=0.5 if request.start_mode == "external" else None,
    )
elif BACKEND == "k2460":
    backend = k2460_sequence_backend(
        VISA_RESOURCE, terminals=TERMINALS, expected_serial_number=EXPECTED_SERIAL
    )
else:
    raise ValidationError(f"Unknown BACKEND {BACKEND!r}; use 'simulator' or 'k2460'")
flyer = SequenceFlyer(backend, name="k2460", start_timeout_s=START_TIMEOUT_S)

## Run
The RunEngine owns this run. The live plot is throttled; the file retains every readout.
Save the executed notebook beside the printed HDF5 path.

In [ ]:
from bluesky import RunEngine
from bluesky.utils import RunEngineInterrupted

from ophyd_electrochemistry.sequences import run_sequence
from ophyd_electrochemistry.sequences.callbacks import LiveSequencePlot, RunFileWriter

RE = RunEngine({}, call_returns_result=True)
writer = RunFileWriter(DATA_DIR)
live_plot = LiveSequencePlot()
RE.subscribe(writer)
RE.subscribe(live_plot)
try:
    outcome = RE(run_sequence(flyer, request)).plan_result
    print(f"Outcome: {outcome}; run file: {writer.path}")
except (RunEngineInterrupted, Exception) as exc:
    print(f"Run interrupted or failed: {type(exc).__name__}: {exc}")
    print(f"Run file: {writer.path}; flyer outcome: {flyer.outcome}")
    print(f"Output confirmed OFF: {flyer.output_enabled is False}")
    raise

## Result
Full resolution from the saved file. Compare actual records with the expected upper bound.

In [ ]:
from collections import Counter

from ophyd_electrochemistry.sequences.callbacks import plot_run_file
from ophyd_electrochemistry.sequences.models import expected_records
from ophyd_electrochemistry.sequences.run_file import read_run_file

result_figure = plot_run_file(writer.path)
result_figure.show()
saved = read_run_file(writer.path)
event_counts = Counter(saved["events"]["kind"])
print(
    "Events: " + ", ".join(f"{kind}={event_counts[kind]}" for kind in ("cutoff", "stop", "abort"))
)
print(f"Final outcome: {saved['run']['outcome']}")
print(
    f"Records written: {len(saved['readout']['t_s'])}; "
    f"expected: {expected_records(request.sequence)}"
)

## If something went wrong

After an abort, inspect the outcome and partial file, then call
`flyer.recover().wait(timeout=5)` before preparing another run. Do not retry an
uncertain hardware command; inspect and confirm output OFF first.

If preparation refuses because the 2460 holds unread records of an earlier
session (e.g. after a crashed kernel), save them first, then run again:

```python
backend.connect()
backend.archive_retained(DATA_DIR / f"retained-{MEASUREMENT_ID}.npz")
backend.close()
```

After a crashed kernel, `read_run_file(path)` reads all flushed records; an
unfinished file retains outcome `incomplete`. For other HDF5 tools, use
`repair_run_file(path, new_path)` to create a separate readable copy. A hard
`RE.halt()` leaves records after the last collect in the backend, outside the file.

Start the next run with a new notebook (replace limits with the session values):

```sh
oe-new-run runs.xlsx example-rest ~/k2460-data \
  --v-min 0 --v-max 3 --i-max 0.01 --max-duration-s 200000 --max-records 1000000
```

The generator validates the whole sheet with those limits and writes them into the
new notebook's `LIMITS`; its `DATA_DIR` is the output directory. Save after running; keep that executed notebook beside its HDF5 file.
